In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv("IPL.csv")
batting_5year = pd.read_csv("Batting_5year.csv")

C:\Users\Dwitiyash\AppData\Local\Temp\ipykernel_22760\3237739085.py:1: DtypeWarning: Columns (0: review_batter, 1: team_reviewed, 2: review_decision, 3: umpire, 4: season, 5: superover_winner, 6: result_type, 7: method, 8: event_match_no) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("IPL.csv")


In [3]:
df_5yr = df[df["year"].between(2021, 2025)].copy()

In [4]:
fours = (
    df_5yr[df_5yr["runs_batter"] == 4]
    .groupby("batter")
    .size()
    .reset_index(name="fours")
)

fours.head()

,batter,fours
0,A Badoni,73
1,A Kamboj,2
2,A Manohar,23
3,A Mhatre,31
4,A Mishra,2


In [5]:
sixes = (
    df_5yr[df_5yr["runs_batter"] == 6]
    .groupby("batter")
    .size()
    .reset_index(name="sixes")
)

sixes.head()

,batter,sixes
0,A Badoni,38
1,A Manohar,14
2,A Mhatre,11
3,A Raghuvanshi,16
4,AB de Villiers,16


In [6]:
dot_balls = (
    df_5yr[
        (df_5yr["runs_batter"] == 0) &
        (df_5yr["valid_ball"] == 1)
    ]
    .groupby("batter")
    .size()
    .reset_index(name="dot_balls")
)

dot_balls.head()

,batter,dot_balls
0,A Badoni,215
1,A Kamboj,6
2,A Manohar,91
3,A Mhatre,41
4,A Mishra,17


In [7]:
boundary = batting_5year[
    ["batter", "runs_5yr", "balls_5yr"]
].copy()

boundary = boundary.merge(fours, on="batter", how="left")
boundary = boundary.merge(sixes, on="batter", how="left")
boundary = boundary.merge(dot_balls, on="batter", how="left")

boundary = boundary.fillna(0)

In [8]:
boundary["boundary_runs"] = (
    boundary["fours"] * 4 +
    boundary["sixes"] * 6
)

In [9]:
boundary["boundary_pct"] = np.where(
    boundary["runs_5yr"] == 0,
    0,
    (boundary["boundary_runs"] /
     boundary["runs_5yr"] * 100)
)

boundary["boundary_pct"] = boundary["boundary_pct"].round(2)

In [10]:
boundary["dot_ball_pct"] = np.where(
    boundary["balls_5yr"] == 0,
    0,
    (boundary["dot_balls"] /
     boundary["balls_5yr"] * 100)
)

boundary["dot_ball_pct"] = boundary["dot_ball_pct"].round(2)

In [11]:
boundary["balls_per_boundary"] = np.where(
    (boundary["fours"] + boundary["sixes"]) == 0,
    np.nan,
    boundary["balls_5yr"] /
    (boundary["fours"] + boundary["sixes"])
)

boundary["balls_per_boundary"] = boundary["balls_per_boundary"].round(2)

In [12]:
boundary = boundary[
    [
        "batter",
        "fours",
        "sixes",
        "dot_balls",
        "boundary_runs",
        "boundary_pct",
        "dot_ball_pct",
        "balls_per_boundary"
    ]
]

In [13]:
boundary.isnull().sum()

batter                 0
fours                  0
sixes                  0
dot_balls              0
boundary_runs          0
boundary_pct           0
dot_ball_pct           0
balls_per_boundary    58
dtype: int64

In [14]:
boundary["batter"].duplicated().sum()

np.int64(0)

In [15]:
boundary.sort_values(
    "boundary_pct",
    ascending=False
).head(10)

,batter,fours,sixes,dot_balls,boundary_runs,boundary_pct,dot_ball_pct,balls_per_boundary
294,Liton Das,1.0,0.0,3.0,4.0,100.00,75.00,4.00
292,A Tomar,1.0,0.0,7.0,4.0,100.00,87.50,8.00
274,KT Maphaka,2.0,0.0,0.0,8.0,100.00,0.00,1.00
260,MA Wood,1.0,1.0,2.0,10.0,90.91,40.00,2.50
265,AA Kulkarni,2.0,0.0,5.0,8.0,88.89,62.50,4.00
242,J Suchith,2.0,1.0,9.0,14.0,87.50,64.29,4.67
237,S Gopal,2.0,1.0,7.0,14.0,87.50,58.33,4.00
89,J Fraser-McGurk,39.0,30.0,82.0,336.0,87.27,42.49,2.80
219,D Wiese,0.0,3.0,5.0,18.0,85.71,45.45,3.67
277,SA Abbott,0.0,1.0,3.0,6.0,85.71,60.00,5.00


In [16]:
boundary.sort_values(
    "balls_per_boundary"
).head(10)

,batter,fours,sixes,dot_balls,boundary_runs,boundary_pct,dot_ball_pct,balls_per_boundary
274,KT Maphaka,2.0,0.0,0.0,8.0,100.00,0.00,1.00
260,MA Wood,1.0,1.0,2.0,10.0,90.91,40.00,2.50
89,J Fraser-McGurk,39.0,30.0,82.0,336.0,87.27,42.49,2.80
105,V Suryavanshi,18.0,24.0,45.0,216.0,85.71,36.89,2.90
158,Urvil Patel,5.0,6.0,11.0,56.0,82.35,34.38,2.91
268,L Wood,0.0,1.0,0.0,6.0,66.67,0.00,3.00
115,R Shepherd,12.0,17.0,28.0,150.0,81.08,32.18,3.00
106,A Mhatre,31.0,11.0,41.0,190.0,79.17,32.28,3.02
162,JG Bethell,9.0,3.0,14.0,54.0,80.60,35.90,3.25
50,TM Head,114.0,47.0,178.0,738.0,78.43,33.84,3.27


In [17]:
boundary.to_csv(
    "boundary_stats.csv",
    index=False
)

print("Boundary Analytics dataset exported successfully!")

Boundary Analytics dataset exported successfully!
